In [364]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import RobustScaler
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize


In [365]:
def load_and_preprocess(path):
    df = pd.read_csv(path)
    print("Original rows:", len(df))

    df = df[~df["code_location"].str.contains("obfuscated", case=False, na=False)].copy()
    print("After removing obfuscated:", len(df))

    df["verdict"] = df["verdict"].astype(int)
    df["is_third_party"] = (~df["code_location"].isin(["developer_written"])).astype(int)
    df["lib_category"] = df["code_location"].apply(
        lambda x: x if x != "developer_written" else "source"
    )

    scaler = RobustScaler()
    df["apk_size_scaled"] = scaler.fit_transform(df[["apk_size"]])

    print("\nVerdict value counts:")
    print(df["verdict"].value_counts())
    
    return df


In [366]:
def compute_tp_stats_and_power(df):
    alpha=0.05
    power_target=0.8

    tp_developer = df.loc[df["is_third_party"] == 0, "verdict"].mean()
    tp_third = df.loc[df["is_third_party"] == 1, "verdict"].mean()
    n_dev = (df["is_third_party"] == 0).sum()
    n_third = (df["is_third_party"] == 1).sum()

    print("Alerts from Developer written codes TP rate:", tp_developer, "|| n =", n_dev)
    print("Alerts from Third-party library codes TP rate:", tp_third, "|| n =", n_third)
    print("Observed difference:", tp_third - tp_developer)

    effect_size = proportion_effectsize(tp_third, tp_developer)
    ratio = n_third / n_dev

    power_analysis = NormalIndPower()

    power = power_analysis.power(
        effect_size=effect_size,
        nobs1=n_dev,
        alpha=alpha,
        ratio=ratio,
        alternative='two-sided'
    )
    print("Post-hoc power current N:", power)

    n_dev_required = power_analysis.solve_power(
        effect_size=effect_size,
        power=power_target,
        alpha=alpha,
        ratio=ratio,
        alternative='two-sided'
    )
    n_third_required = n_dev_required * ratio

    print("Required n_dev for 80% power:", n_dev_required)
    print("Required n_third for 80% power:", n_third_required)



In [367]:
def downsample_to_balanced(df):
    majority = df[df["is_third_party"] == 1]
    minority = df[df["is_third_party"] == 0]
    n_min = len(minority)

    down_sample = majority.sample(n=n_min, replace=False, random_state=42)

    df_bal = pd.concat([down_sample, minority], axis=0)
    df_bal = df_bal.sample(frac=1, random_state=42).reset_index(drop=True)

    print("\nBalanced is_third_party counts:")
    print(df_bal["is_third_party"].value_counts())

    return df_bal

In [368]:
df = load_and_preprocess("alerts_with_category_with_apk_size_updated.csv")
print("\n------ Power analysis on full (unbalanced) data ------")
compute_tp_stats_and_power(df)

df = downsample_to_balanced(df)
print("\n------ Power analysis on balanced data ------")
compute_tp_stats_and_power(df)

Original rows: 6140
After removing obfuscated: 5612

Verdict value counts:
verdict
0    2898
1    2714
Name: count, dtype: int64

------ Power analysis on full (unbalanced) data ------
Alerts from Developer written codes TP rate: 0.6908023483365949 || n = 511
Alerts from Third-party library codes TP rate: 0.46285042148598315 || n = 5101
Observed difference: -0.22795192685061177
Post-hoc power current N: 0.9999999999999997
Required n_dev for 80% power: 39.78181369165043
Required n_third for 80% power: 397.1174787497238

Balanced is_third_party counts:
is_third_party
0    511
1    511
Name: count, dtype: int64

------ Power analysis on balanced data ------
Alerts from Developer written codes TP rate: 0.6908023483365949 || n = 511
Alerts from Third-party library codes TP rate: 0.4598825831702544 || n = 511
Observed difference: -0.23091976516634055
Post-hoc power current N: 0.999999988132514
Required n_dev for 80% power: 70.50555139087358
Required n_third for 80% power: 70.50555139087358
